# Customer Churn Classification

This notebook audits the benchmark, runs the reproducible comparison, inspects classification evidence, and reconciles the selected model with exported artifacts.

## 1. Setup and dataset audit

The churn target is excluded from the feature matrix. The split is stratified because the positive class is a minority.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'customer_churn.csv'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
df = pd.read_csv(DATA_PATH)
audit = {'shape': df.shape, 'missing_values': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()), 'churn_rate': round(float(df['churn'].mean()), 4)}
display(df.head())
audit

## 2. Execute the reproducible experiment

The command regenerates the comparison table, figures, permutation importance, and JSON summary.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / 'run_experiment.py'), '--data', str(DATA_PATH), '--output', str(OUTPUT_DIR)], check=True)

## 3. Model comparison and visual evidence

PR-AUC is the primary selection metric because the churn class is imbalanced. Accuracy alone is not sufficient.

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'model-comparison.csv')
display(comparison)
display(Image(filename=str(OUTPUT_DIR / 'model-comparison.png')))
display(Image(filename=str(OUTPUT_DIR / 'selected-confusion-matrix.png')))
display(Image(filename=str(OUTPUT_DIR / 'permutation-importance.png')))

## 4. Reconcile outputs

The JSON file is the authoritative record for the selected model, metrics, artifacts, and limitations.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'results-summary.json').read_text())
assert summary['selected_model'] == comparison.iloc[0]['model']
summary

## Interpretation

The model estimates statistical churn risk, not customer intent or causation. A real deployment would require calibration, fairness review, intervention-cost analysis, drift monitoring, and human oversight.